# Causal Analysis Report

Comprehensive causal discovery and validation for the Azimuth pipeline.  
Inspired by the analysis structure of [juangamella/causal-chamber-paper](https://github.com/juangamella/causal-chamber-paper).

## Contents

1. **Ground Truth & Attention-based Discovery** — Extract the inter-process DAG and compare with CausaliT attention weights
2. **Causal Discovery Validation** — Precision / Recall / F1 / SHD against ground truth, with optional GES/PC baselines
3. **Interventional Validation** — Verify causal effects via `SCM.do()` and p-value matrix (Appendix V style)
4. **Out-of-Distribution Analysis** — Robustness under distributional shift
5. **Symbolic Regression** — Recover structural equations from data

In [ ]:
import sys
from pathlib import Path

# Ensure the repository root is on the path
REPO_ROOT = Path().resolve().parent.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 9,
    'axes.titlesize': 10,
})

print(f'Repository root: {REPO_ROOT}')

## 0. Load Datasets

In [ ]:
from scm_ds.datasets import (
    ds_scm_laser,
    ds_scm_plasma,
    ds_scm_galvanic,
    ds_scm_microetch,
)

datasets = {
    'laser': ds_scm_laser,
    'plasma': ds_scm_plasma,
    'galvanic': ds_scm_galvanic,
    'microetch': ds_scm_microetch,
}

PROCESS_ORDER = ['laser', 'plasma', 'galvanic', 'microetch']

for name, ds in datasets.items():
    print(f'{name:12s}  inputs={ds.input_labels}  outputs={ds.target_labels}')

---
## 1. Ground Truth & Attention-based Discovery

In [ ]:
from causaliT.causal_discovery.ground_truth import (
    extract_ground_truth_dag,
    get_observable_variables,
)

obs_vars = get_observable_variables(datasets, PROCESS_ORDER, include_F=True)
print('Observable variables:', obs_vars)

gt_dag = extract_ground_truth_dag(datasets, PROCESS_ORDER, include_F=True)
print(f'\nGround truth DAG shape: {gt_dag.shape}')
gt_dag

In [ ]:
# Visualise the ground-truth adjacency matrix
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    gt_dag, annot=True, fmt='d', cmap='Blues',
    xticklabels=gt_dag.columns, yticklabels=gt_dag.index,
    linewidths=0.5, ax=ax,
)
ax.set_title('Ground Truth Inter-Process DAG (row <- col)')
plt.tight_layout()
plt.show()

### 1b. Attention-based Discovery (requires trained CausaliT checkpoint)

Uncomment and adapt the cell below once a trained checkpoint is available.

In [ ]:
# from causaliT.causal_discovery.attention_discovery import (
#     AttentionGraphExtractor, load_vars_maps,
# )
#
# # Load model and variable maps
# # model = ...  # Load your trained ProT / CausaliT checkpoint
# # iv_map, tv_map = load_vars_maps('path/to/generated_data/')
#
# # extractor = AttentionGraphExtractor(model, iv_map, tv_map)
# # extractor.register_hooks()
#
# # attn_matrices = extractor.collect_attention(dataloader, max_batches=50)
# # var_attn = extractor.attention_to_variable_graph(attn_matrices)
# # est_adj = AttentionGraphExtractor.threshold_to_adjacency(var_attn, threshold=0.1)
#
# # fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
# # for ax, (data, title, cmap) in zip(axes, [
# #     (gt_dag, 'Ground Truth', 'Blues'),
# #     (var_attn, 'Attention Weights', 'Reds'),
# #     (est_adj, 'Estimated DAG', 'Greens'),
# # ]):
# #     sns.heatmap(data, annot=True, fmt='.2f', cmap=cmap, ax=ax)
# #     ax.set_title(title)
# # plt.tight_layout()
# # plt.show()
print('Attention-based discovery section ready (requires trained checkpoint).')

---
## 2. Causal Discovery Validation

In [ ]:
from causaliT.causal_discovery.discovery_validation import DiscoveryValidator

validator = DiscoveryValidator(datasets, PROCESS_ORDER)
print(f'Ground truth shape: {validator.ground_truth.shape}')
print(f'Observable vars: {validator.obs_vars}')

# Generate i.i.d. data for classical baselines
iid_data = validator.generate_iid_data(n_samples=5000, seed=42)
print(f'\nGenerated i.i.d. data: {iid_data.shape}')
iid_data.describe()

In [ ]:
# Run classical baselines (GES, PC) if causal-learn is installed
try:
    classical_results = validator.validate_classical_baselines(
        iid_data, run_ges=True, run_pc=True, pc_alpha=0.05
    )
    for algo, res in classical_results.items():
        if 'error' not in res:
            print(f'\n{algo} results:')
            for k, v in res.items():
                if k != 'adjacency':
                    print(f'  {k}: {v}')
        else:
            print(f'{algo}: {res["error"]}')
except Exception as e:
    print(f'Classical baselines skipped: {e}')

---
## 3. Interventional Validation

In [ ]:
from causaliT.causal_discovery.interventional_analysis import InterventionalAnalyzer

interventional = InterventionalAnalyzer(datasets, PROCESS_ORDER)

# Example: do(PowerTarget = 0.8)
dist_cmp = interventional.compare_distributions(
    var='PowerTarget', value=0.8, n_samples=5000, seed=42
)
print('Distribution comparison for do(PowerTarget=0.8):')
dist_cmp[['mean_shift', 'ks_statistic', 'ks_pvalue']]

In [ ]:
# P-value matrix (Appendix V style)
pval_matrix, sig_matrix = interventional.compute_pvalue_matrix(
    n_samples=5000, seed=42, significance=0.05
)

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(
    -np.log10(pval_matrix + 1e-20),
    cmap='YlOrRd', ax=ax, annot=True, fmt='.1f',
    xticklabels=pval_matrix.columns,
    yticklabels=pval_matrix.index,
)
ax.set_title('Interventional p-value matrix (-log10 scale)')
ax.set_ylabel('Intervention variable')
ax.set_xlabel('Response variable')
plt.tight_layout()
plt.show()

print('\nSignificant effects (p < 0.05):')
sig_matrix

---
## 4. Out-of-Distribution Analysis

In [ ]:
from causaliT.causal_discovery.ood_analysis import (
    OODAnalyzer, DEFAULT_SHIFTS,
)

ood = OODAnalyzer(datasets, PROCESS_ORDER)

# Run distributional comparison for all default shifts
ood_results = ood.run_full_ood_analysis(
    shifts=DEFAULT_SHIFTS, n_samples=5000, seed=42
)

for desc, df in ood_results.items():
    print(f'\n--- {desc} ---')
    print(df[['id_mean', 'ood_mean', 'mean_shift', 'ks_pvalue']].to_string())

In [ ]:
# Visualise the shift magnitudes
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, (desc, df) in zip(axes.flat, ood_results.items()):
    shifted = df['ks_pvalue'] < 0.05
    colors = ['red' if s else 'steelblue' for s in shifted]
    ax.barh(range(len(df)), df['mean_shift'].values, color=colors)
    ax.set_yticks(range(len(df)))
    ax.set_yticklabels(df.index, fontsize=7)
    ax.set_title(desc, fontsize=8)
    ax.set_xlabel('Mean shift')
    ax.axvline(0, color='k', linewidth=0.5)
plt.tight_layout()
plt.show()

---
## 5. Symbolic Regression

In [ ]:
from causaliT.causal_discovery.symbolic_analysis import SymbolicAnalyzer

symbolic = SymbolicAnalyzer(datasets, PROCESS_ORDER)

# Run polynomial baseline for all processes
sym_results = symbolic.run_full_analysis(
    n_samples=10000, seed=42, use_pysr=False, max_poly_degree=4
)

display_cols = [c for c in sym_results.columns if c != 'ground_truth_expr']
sym_results[display_cols]

In [ ]:
# Show ground truth vs discovered expressions
for _, row in sym_results.iterrows():
    proc = row['process']
    out_var = row['output_var']
    gt = row.get('ground_truth_expr', 'N/A')
    poly = row.get('poly_expression', 'N/A')
    r2 = row.get('poly_r2_test', 'N/A')
    print(f'\n{proc} / {out_var}:')
    print(f'  Ground truth : {gt}')
    print(f'  Polynomial   : {poly}')
    print(f'  R^2 (test)   : {r2}')

---
## 6. Generate PDF Report

Assembles all results into a publication-ready PDF.

In [ ]:
from causaliT.causal_discovery.report_generator import CausalAnalysisReportGenerator

output_dir = REPO_ROOT / 'causaliT' / 'causal_discovery' / 'reports'
output_dir.mkdir(exist_ok=True)

report = CausalAnalysisReportGenerator(output_dir / 'causal_analysis_report.pdf')
report.add_title()

# Section 1: Ground Truth
report.add_ground_truth_section(gt_dag)

# Section 3: Interventional
intervention_results = interventional.run_all_interventions(
    intervention_specs=[('PowerTarget', 0.8), ('RF_Power', 300.0)],
    n_samples=5000,
)
report.add_interventional_section(intervention_results, pval_matrix)

# Section 4: OOD
report.add_ood_section(ood_results)

# Section 5: Symbolic
report.add_symbolic_section(sym_results)

# Build the PDF
report_path = report.generate()
print(f'Report saved to: {report_path}')

---
*End of Causal Analysis Notebook*